# 实践 10.2  ChIP-seq/ATAC-seq命令模板与TSS质控

《统计遗传与智能育种学》第十章 · 电子资源 `10b_chip_atac_toolchain.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：10b_chip_atac_toolchain.ipynb。实践以FastQC/Trimmomatic、Bowtie2、SAMtools、Picard、MACS2和deepTools组成分析命令模板，同时用虚拟片段坐标实现去重、峰富集、转录起始位点窗口计数和重复相关的可运行演示。其中ATAC的--nomodel、--shift和--extsize仅作为示例参数；实际设置取决于MACS版本、单/双端信息以及是否已进行Tn5位移处理。

真实流程：原始测序读段质控/接头处理 → Bowtie2 → SAMtools/Picard → MACS2 → deepTools/整合基因组浏览器（Integrative Genomics Viewer, IGV） → TSS富集与重复一致性  
宽组蛋白标记可使用适合宽峰的模型；ATAC参数必须与数据结构和软件版本匹配。

运行检查：重启内核并运行全部单元  
预期：查看正式分析的命令模板并运行模拟片段代码，输出重复片段比例、30个峰的重复相关、峰富集倍数及TSS窗口计数，说明这些示例统计量与正式工具结果的差别。

## 输入、完整代码、实际输出与解释

输入与可复现条件：正式分析命令模板需要真实FASTQ、参考基因组索引、注释和相应软件；本章可运行核心使用虚拟比对后片段，SEED=1002，无需外部数据。

**正式分析命令模板（需真实FASTQ与相应软件；不作为本章模拟的可运行验收核心）**（外部工具代码模板，本 notebook 不执行）

```bash
fastqc sample_R1.fastq.gz sample_R2.fastq.gz
trimmomatic PE sample_R1.fastq.gz sample_R2.fastq.gz clean_R1.fq.gz unpaired_R1.fq.gz clean_R2.fq.gz unpaired_R2.fq.gz ILLUMINACLIP:adapters.fa:2:30:10 SLIDINGWINDOW:4:20 MINLEN:30
bowtie2 -x genome -1 clean_R1.fq.gz -2 clean_R2.fq.gz | samtools sort -o sample.bam
picard MarkDuplicates I=sample.bam O=sample.dedup.bam M=dup_metrics.txt REMOVE_DUPLICATES=true
macs2 callpeak -t sample.dedup.bam -f BAMPE -g genome_size -n sample -q 0.01
multiBamSummary bins --bamfiles rep1.bam rep2.bam -o replicate.npz
computeMatrix reference-point --referencePoint TSS -b 2000 -a 2000 -R genes.bed -S sample.bw -o tss.mat.gz
plotProfile -m tss.mat.gz -out tss_profile.pdf
# ATAC单端/特定处理情形才考虑示例：--nomodel --shift -75 --extsize 150
```

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=1002
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)

# 虚拟“比对后片段”：重复片段、peak富集、TSS profile和重复相关性
genome=1_000_000; tss=np.array([100000,300000,700000]); other=np.linspace(30000,970000,27,dtype=int); peaks=np.unique(np.r_[tss,other]); peak_weights=rng.lognormal(0,.7,len(peaks)); peak_weights/=peak_weights.sum()
def fragments(n=5000):
 start=rng.integers(0,genome,n); enrich=rng.random(n)<.35; start[enrich]=rng.choice(peaks,enrich.sum(),p=peak_weights)+rng.normal(0,350,enrich.sum()).astype(int); length=rng.integers(50,300,n); return np.column_stack([start,np.clip(start+length,0,genome)])
f1=fragments(8000); dup=np.vstack([f1,f1[rng.choice(len(f1),1000)]]); dedup=np.unique(dup,axis=0); print('raw fragments',len(dup),'deduplicated',len(dedup),'duplicate fraction',round(1-len(dedup)/len(dup),3))
def peak_counts(fr): return np.array([np.sum(np.abs(fr[:,0]-p)<800) for p in peaks])
f2=fragments(8000); print('candidate peaks',len(peaks),'peak-count replicate correlation',round(np.corrcoef(peak_counts(dedup),peak_counts(f2))[0,1],3)); near=sum(np.sum(np.abs(dedup[:,0]-p)<800) for p in peaks); expected_random=len(dedup)*len(peaks)*1600/genome; print('peak enrichment over random expectation',round(near/max(expected_random,1e-8),2))
for dist in [250,500,1000,2000]: print('TSS ±',dist,'fragment starts',int(sum(np.sum(np.abs(dedup[:,0]-x)<=dist) for x in tss)))


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 1002
raw fragments 9000 deduplicated 8000 duplicate fraction 0.111
candidate peaks 30 peak-count replicate correlation 0.99
peak enrichment over random expectation 7.89
TSS ± 250 fragment starts 242
TSS ± 500 fragment starts 362
TSS ± 1000 fragment starts 448
TSS ± 2000 fragment starts 498


结果解释：虚拟数据中重复片段比例约0.111；30个候选峰上的重复计数相关约0.990；峰窗口的片段起点计数为均匀背景期望的约7.89倍。TSS窗口计数从±250 bp的242逐步增加到±2000 bp的498，这些是累计窗口计数，未计算以侧翼归一化的TSS富集分数。

可修改练习：把enrich概率从0.35改为0.10并重新运行，比较峰窗口富集、重复相关和TSS窗口计数；结合本节FRiP定义，说明固定阈值的适用条件。